# P102 — Algoritmos de optimización proximal de políticas

## 1. Título y paper

**Paper:** *Proximal Policy Optimization Algorithms*  
**Autoría:** John Schulman, Filip Wolski, Prafulla Dhariwal, Alec Radford, Oleg Klimov  
**Año y venue:** 2017 · arXiv:1707.06347  
**Nivel:** L3 · **Motor:** `ppo`  
**Ficha completa:** [`P102_ppo`](../../papers/foundational/P102_ppo/README.md)

**Hito:** Consigue la estabilidad de TRPO con una función objetivo que se implementa en unas líneas y se optimiza con descenso de gradiente corriente.

- [arXiv:1707.06347](https://arxiv.org/abs/1707.06347)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: En gradiente de políticas, un paso demasiado grande destruye la política: se vuelve casi determinista, deja de explorar y no puede recuperarse. TRPO lo resolvía con una restricción de divergencia KL, a costa de una optimización de segundo orden compleja.
2. Ejecutar una implementación mínima de la propuesta: Sustituir la restricción por un **recorte** del cociente de probabilidades entre la política nueva y la vieja. Pasado el umbral, mejorar más no aporta al objetivo, así que el gradiente deja de empujar. Sin restricciones, sin segundo orden.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P26
- Schulman et al. (2015), TRPO
- P101


## 4. Intuición

En gradiente de políticas, un lote con una estimación exagerada puede empujar la política hasta volverla determinista. A partir de ahí deja de explorar y no puede rectificar. PPO lo evita con un truco de una línea: si la política nueva se aleja demasiado de la vieja, el objetivo deja de premiar el cambio.


## 5. Concepto mínimo

```text
r = π_nueva(a|s) / π_vieja(a|s)          ← cuánto ha cambiado la política

L = mín( r·A ,  clip(r, 1−ε, 1+ε)·A )

Con A > 0 y r > 1+ε  →  el objetivo se queda plano: el gradiente deja de empujar
Sin restricciones. Sin segundo orden. Sin divergencia KL que calcular.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('ppo', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuánto vale el objetivo sin recorte a ratio 5,0? ¿Y con recorte?
2. ¿Cuánto puede moverse la probabilidad en un solo paso?
3. ¿Qué pasa cuando la acción buena cambia a mitad del entrenamiento?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('ppo', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('ppo', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

A ratio 5,0 el objetivo sin recorte vale **6,0** y el de PPO, **1,44** — el mismo valor que a ratio 1,2. El salto máximo de probabilidad en un paso es **0,4382** sin recorte y **0,1727** con él. Y tras el giro del entorno, la política sin recorte acaba en 0,9412 y la recortada en **0,6909**.


## 10. Comentario pedagógico

Acotar el paso no es prudencia: es lo que conserva la capacidad de rectificar. Una política saturada tiene gradiente casi nulo —la derivada de la sigmoide se anula en los extremos— así que salir de ahí cuesta muchísimos pasos. Es el mismo motivo por el que [RLHF](../../papers/foundational/P12_instructgpt/README.md) penaliza alejarse del modelo base.


## 11. Error o anti-patrón deliberado

Anti-patrón: creer que el recorte garantiza mejora monótona.


In [ ]:
print('TRPO tenia una garantia teorica de mejora monotona bajo sus supuestos.')
print('PPO la cambia por simplicidad: NO hereda esa garantia.')
print('Se justifica por rendimiento empirico, y eso hay que decirlo al citarlo.')

## 12. Corrección

Lo que el recorte sí hace, medido:


In [ ]:
r = run_paper_lab('ppo', seed=7)['result']
print('salto maximo de p:', r['salto_maximo_de_probabilidad_en_un_paso'])
print('pasos saturados  :', r['pasos_saturados'])
print('p al final       :', r['p_al_final'])
for fila in r['comparacion_del_objetivo']:
    print(f"  ratio {fila['ratio']:<5} sin recorte {fila['sin_recorte']:<7} con recorte {fila['con_recorte']}")

## 13. Desafío guiado

Explica por qué el objetivo recortado usa el MÍNIMO entre las dos expresiones, y qué pasaría con ventaja negativa si usara el máximo.


In [ ]:
r = run_paper_lab('ppo', seed=3)['result']
show(r)

## 14. Desafío autónomo

Entrena PPO sobre un entorno de control clásico barriendo epsilon entre 0,05 y 0,5. Documenta la frontera entre aprender demasiado despacio y colapsar.


## 15. Evidencia de aprendizaje

Guarda la comparación del objetivo con y sin recorte y tu explicación de por qué una política saturada no se recupera.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P102_ppo/README.md) · evaluación formal: [`assessments/papers/P102_ppo.md`](../../assessments/papers/P102_ppo.md)


## 16. Cierre

El control ya se puede aprender. Pero entrenar sobre hardware real es lento, caro y peligroso: hay que entrenar en simulación, y eso abre otro hueco.


## 17. Conexión con el siguiente hito

- P12
- P22

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
